In [ ]:
import glob
import pandas as pd

bicdycle = sorted(glob.glob("../data/processed_data/bicycle_sample.csv"))

In [ ]:
# 1. 아침 출근 시간대(7~9시) & 평일(근무일) 데이터만 필터링
# 이미 만들어두신 is_commute와 is_workday를 활용합니다.
morning_rush = bicycle[(bicycle['start_hour'].between(7, 9)) & (bicycle['is_workday'] == True)]

# 2. 대여소별 대여 건수 (출발)
rent_cnt = morning_rush.groupby('시작_대여소명')['전체_건수'].sum().reset_index(name='대여량')
rent_cnt.rename(columns={'시작_대여소명': '대여소명'}, inplace=True)

# 3. 대여소별 반납 건수 (종료)
return_cnt = morning_rush.groupby('종료_대여소명')['전체_건수'].sum().reset_index(name='반납량')
return_cnt.rename(columns={'종료_대여소명': '대여소명'}, inplace=True)

# 4. 데이터 병합 및 '불균형 지수' 계산
# 불균형 지수 = 반납량 - 대여량
balance_df = pd.merge(rent_cnt, return_cnt, on='대여소명', how='outer').fillna(0)
balance_df['불균형지수'] = (balance_df['반납량'] - balance_df['대여량'])/(balance_df['반납량'] + balance_df['대여량'])

# 5. 결과 확인 (극단적인 불균형 Top 10)
print("🔴 아침 출근길, 자전거가 쏟아져 들어와 거치대가 부족한 곳 (반납 폭발 / 오피스 밀집)")
display(balance_df.sort_values(by='불균형지수', ascending=False).head(10))

print("\n🔵 아침 출근길, 자전거가 다 빠져나가서 탈 게 없는 곳 (대여 폭발 / 주거지 밀집)")
display(balance_df.sort_values(by='불균형지수', ascending=True).head(10))